### PRM / Reward Model Memory Sizes
Loads each PRM/reward model with HuggingFace transformers and measures GPU memory used.

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import gc

import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig

In [ ]:
base_dir = '/groups/chichengz/tnn/datasets/'

llm_dir = base_dir + "Qwen2.5-Math-PRM-7B"
# llm_dir = base_dir + "Llama3.1-8B-PRM-Deepseek-Data"
# llm_dir = base_dir + "Skywork-Reward-V2-Llama3.2-3B"
# llm_dir = base_dir + "Skywork-Reward-V2-Llama3.2-8B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(llm_dir, trust_remote_code=True)

config = AutoConfig.from_pretrained(llm_dir, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.pad_token = tokenizer.eos_token

config.pad_token_id = tokenizer.pad_token_id

llm_tf = AutoModel.from_pretrained(
    llm_dir,
    config=config,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
llm_tf.eval()

gc.collect()
torch.cuda.empty_cache()
free_memory, total_memory = torch.cuda.mem_get_info(0)
print(f'#--- memory: {(total_memory - free_memory) / (1024**3):.2f} GB')